# main regression

In [1]:
# ==========================================================
#  21 Industries: CLEAN stacked DDD Event Studies
#  + Long-run control: Δlog(min_wage) relative to pre-2023 base
#  (FIXED: compute wage_base from policy-only rows BEFORE dropping employment_rate)
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------------
# 1. Read and clean data
# -----------------------------
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
raw = pd.read_csv(fr"{base}\14100355.csv")

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

# Convert to numeric
# (min_wage may exist in 2022; employment_rate may be missing in 2022)
raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

# Monthly time index
raw["m_id"] = raw["date"].dt.year * 12 + raw["date"].dt.month

# -----------------------------
# 2. Construct wage_base using the "policy series" only
#    For each province, take the last observed min_wage before 2023-01
#    (does not depend on employment_rate)
# -----------------------------
policy = raw.dropna(subset=["min_wage"]).copy()
policy = policy.sort_values(["province", "date"])

base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
          .sort_values(["province", "date"])
          .groupby("province", as_index=False)
          .tail(1)[["province", "min_wage"]]
          .rename(columns={"min_wage": "wage_base"})
)

# Merge back to raw
# (all industries/months obtain dlogW)
raw = raw.merge(base_wage, on="province", how="left")

# ΔlogW: relative to the pre-2023 base
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

# -----------------------------
# 3. Generate regression-ready dataframe:
#    must have employment_rate, min_wage, and dlogW
# -----------------------------
df = raw.dropna(subset=["employment_rate", "min_wage", "dlogW"]).copy()
df = df.sort_values(["province", "date"])

# -----------------------------
# 4. Identify 2023+ minimum wage events & doses (province-level)
# -----------------------------
df["dW"]   = df.groupby("province")["min_wage"].diff()
df["lagW"] = df.groupby("province")["min_wage"].shift(1)
df["dose"] = (df["dW"] / df["lagW"]) * 100

events = (
    df.loc[
        (df["dose"] > 0) & (df["date"] >= "2023-01-01"),
        ["province", "date", "dose", "m_id"]
    ]
    .drop_duplicates()
    .rename(columns={"date": "t0", "m_id": "m0"})
    .reset_index(drop=True)
)

print(f"Identified 2023+ MW increase events: {len(events)}")

# -----------------------------
# 5. Event-window parameters
# -----------------------------
L, R = 2, 3
BASE_K = -1
event_var = lambda k: f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"

# -----------------------------
# 6. Global "hole-punching": clean controls (based on df)
# -----------------------------
df["in_any_window"] = 0

for prov, g in events.groupby("province"):
    m0_list = g["m0"].tolist()
    idx = df["province"].eq(prov)
    m = df.loc[idx, "m_id"].values
    hit = np.zeros_like(m, dtype=bool)
    for m0 in m0_list:
        hit |= (m >= m0 - L) & (m <= m0 + R)
    df.loc[idx, "in_any_window"] = hit.astype(int)

# -----------------------------
# 7. Industry-by-industry stacked DDD
#    (short-run event-time effects + long-run dlogW)
# -----------------------------
industries = sorted(df["industry"].unique())
results = []
all_stacks = []   # Store stacks (for diagnostics / placebo alignment)

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    stack_list = []

    # ---------- Loop over events ----------
    for ev in events.itertuples(index=False):
        ev_prov = ev.province
        m0      = ev.m0
        dose    = ev.dose

        # ---- Treated: event window for the treated province ----
        treated = dfi[dfi["province"] == ev_prov].copy()
        treated["EventTime"] = (treated["m_id"] - m0).astype(int)

        treated = treated[
            (treated["EventTime"] >= -L) &
            (treated["EventTime"] <= R)
        ].copy()

        # Drop overlapping windows from other events in the same province
        other_m0 = events.loc[
            (events["province"] == ev_prov) & (events["m0"] != m0),
            "m0"
        ].tolist()

        if other_m0:
            m = treated["m_id"].values
            overlap = np.zeros_like(m, dtype=bool)
            for om0 in other_m0:
                overlap |= (m >= om0 - L) & (m <= om0 + R)
            treated = treated[~overlap].copy()

        # Event-time × dose (baseline k = -1 omitted)
        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            treated[event_var(k)] = (
                (treated["EventTime"] == k).astype(int) * dose
            )

        if treated.empty:
            continue

        # ---- Controls: other provinces & not in any window ----
        controls = dfi[
            (dfi["province"] != ev_prov) &
            (dfi["in_any_window"] == 0)
        ].copy()

        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            controls[event_var(k)] = 0.0

        stack_list.append(
            pd.concat([treated, controls], ignore_index=True)
        )

    if not stack_list:
        continue

    # ---------- Stack construction ----------
    stack = pd.concat(stack_list, ignore_index=True)
    stack["industry"] = ind
    all_stacks.append(stack.copy())

    # ---------- Valid event columns ----------
    event_cols = [c for c in stack.columns if c.startswith("event_")]
    valid_events = [c for c in event_cols if stack[c].sum() != 0]
    if not valid_events:
        continue

    # ---------- Main regression ----------
    formula = f"""
    employment_rate ~
        dlogW
        + {' + '.join(valid_events)}
        + C(province) + C(date)
    """

    model = smf.ols(formula, data=stack).fit(
        cov_type="cluster",
        cov_kwds={"groups": stack["province"]}
    )

    # ---------- Event-time coefficients ----------
    for v in valid_events:
        tag  = v.split("_")[1]      # m2 / p3
        sign = tag[0]
        num  = int(tag[1:])
        k    = -num if sign == "m" else num

        results.append({
            "industry": ind,
            "k": k,
            "coef": model.params.get(v, np.nan),
            "p_value": model.pvalues.get(v, np.nan),
            "n_obs": int(model.nobs)
        })

    # ---------- Long-run effect ----------
    results.append({
        "industry": ind,
        "k": "long",
        "coef": model.params.get("dlogW", np.nan),
        "p_value": model.pvalues.get("dlogW", np.nan),
        "n_obs": int(model.nobs)
    })

    # ---------- ★ JOINT Wald test (all event-time coefficients = 0) ----------
    hypotheses = " = 0, ".join(valid_events) + " = 0"
    wald = model.wald_test(hypotheses, scalar=True)

    results.append({
        "industry": ind,
        "k": "JOINT",
        "coef": wald.statistic,
        "p_value": wald.pvalue, 
        "n_obs": int(model.nobs)
    })

# -----------------------------
# 8. Output and save main regression results
# -----------------------------
res_df = (
    pd.DataFrame(results)
      .sort_values(["industry", "k"])
)

# ---------- Save CSV ----------
out_path = r"C:\Users\SC2zh\Desktop\S4 paper\results_main_ddd.csv"
res_df.to_csv(out_path, index=False)
print(f"\nMain regression results saved to: {out_path}")

# ---------- Print ----------
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n=== CLEAN stacked DDD + Long-run ΔlogW control (FIXED) ===")
print(res_df.to_string(index=False))

sig_df = res_df[res_df["p_value"] < 0.10]
print("\n=== Significant Results (p < 0.10) ===")
print(sig_df.to_string(index=False) if not sig_df.empty else "None")


Identified 2023+ MW increase events: 23

Main regression results saved to: C:\Users\SC2zh\Desktop\S4 paper\results_main_ddd.csv

=== CLEAN stacked DDD + Long-run ΔlogW control (FIXED) ===
                                           industry     k    coef  p_value  n_obs
                    Accommodation and food services    -2 -0.0014   0.3903   4209
                    Accommodation and food services     0  0.0004   0.5526   4209
                    Accommodation and food services     1 -0.0003   0.8155   4209
                    Accommodation and food services     2  0.0017   0.3694   4209
                    Accommodation and food services     3  0.0003   0.9041   4209
                    Accommodation and food services JOINT 48.6972   0.0000   4209
                    Accommodation and food services  long -0.1204   0.0091   4209
                                        Agriculture    -2  0.0008   0.8686   4209
                                        Agriculture     0 -0.0076   0.0714